# SOCIOWEAVE — T4 small-model core execution smoke

This notebook exercises the local small-model core pipeline on a single operator-supplied prompt: q=3 requirement extraction, extraction aggregation, typed decomposition parsing, structural checks V1/V2/V5, metric calculation, and the decision operators. It uses Qwen2.5-1.5B-Instruct in 4-bit mode by default.

**Scientific boundary:** this is an engineering smoke test. It is not E1–E9, does not reproduce the paper's MA/MB/MC/ME model matrix, does not evaluate heterogeneous agents, and does not produce benchmark evidence. Use approved frozen benchmark snapshots and the separately preregistered campaign for scientific experiments.

In [ ]:
import pathlib, subprocess, sys, json, torch, platform
REPO_URL = 'https://github.com/msellamiTN/socioweave-studio.git'
CODE_SHA = '749f97d3d3f807e78c0e8eaa0bf8cb134d63a649'  # immutable small-model runtime revision
ROOT = pathlib.Path('/content/socioweave-t4-core-smoke')
REPO = ROOT / 'repo'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
if not REPO.exists(): subprocess.check_call(['git','clone','--no-checkout',REPO_URL,str(REPO)])
subprocess.check_call(['git','-C',str(REPO),'fetch','origin',CODE_SHA])
subprocess.check_call(['git','-C',str(REPO),'checkout','--detach',CODE_SHA])
actual_sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert actual_sha == CODE_SHA, (actual_sha, CODE_SHA)
subprocess.check_call([sys.executable,'-m','pip','install','-e',str(REPO / 'engine') + '[local,dev]'])
print('Pinned SHA:', actual_sha, '| Python:', platform.python_version())

In [ ]:
assert torch.cuda.is_available(), 'Select a Colab GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert 'T4' in gpu_name, f'Expected T4, got {gpu_name}'
assert gpu_gib >= 14.0, f'Unexpected memory capacity: {gpu_gib:.2f} GiB'
print('GPU:', gpu_name, '| GiB:', round(gpu_gib,2), '| CUDA:', torch.version.cuda)
print(subprocess.check_output(['nvidia-smi'],text=True))

## 1. Configure the small model and prompt

Edit `PROMPT_TEXT` to run the same pipeline on another prompt. The small model is deliberately homogeneous across the three extractors.

In [ ]:
from sociodec.small_model_runtime import SmallModelConfig, build_t4_provider, run_small_model_core
MODEL_CONFIG = SmallModelConfig(model_id='Qwen/Qwen2.5-1.5B-Instruct', load_in_4bit=True, extractor_count=3, seed=17)
PROMPT_TEXT = 'Design a permit application service. It must protect applicant data, support online submission, and provide an audit report.'
provider = build_t4_provider(MODEL_CONFIG)
run = run_small_model_core(PROMPT_TEXT, provider=provider, config=MODEL_CONFIG)
core_report = run.report()
print(json.dumps(core_report, ensure_ascii=False, indent=2))
assert core_report['provider_calls'] == 4
assert core_report['subproblems'] >= 1

## 2. Exercise the metric and decision core

The following checks operate on the model-generated decomposition. They test code paths, not benchmark scores or claims about strategy quality.

In [ ]:
from sociodec.metrics import decomposition_metrics
from sociodec.decisions import Candidate, authority, rank_sum, majority, union, synthesis, compromise, consensus_medoid
d = run.decomposition
metric_result = decomposition_metrics(
    requirements=run.validation.effective_requirements,
    covered_requirements=d.cov,
    subproblem_items=d.subproblems,
    edges=d.E,
    acyclic=(run.validation.v2_acyclicity.status.value == 'pass'),
    composition_complete=(run.validation.v5_composition.status.value == 'pass'),
    depth=max(1, len(d.subproblems)),
)
a, b = Candidate('candidate-a', d), Candidate('candidate-b', d)
all_items = tuple(d.subproblems)
ballots = {'voter-1': all_items, 'voter-2': all_items, 'voter-3': all_items}
decision_results = {
    'authority': authority([a,b], judge_id='smoke-judge', selected_candidate_id='candidate-a', rationale='selected supplied candidate'),
    'rank_sum': rank_sum([a,b], {'voter-1':['candidate-a','candidate-b'], 'voter-2':['candidate-b','candidate-a']}),
    'majority': majority([a,b], ballots),
    'union': union([a,b]),
    'synthesis': synthesis([a,b], d, synthesizer_id='smoke-synthesizer', rationale='explicit supplied decomposition'),
    'compromise': compromise([a,b], d, mediator_id='smoke-mediator', rationale='explicit supplied decomposition'),
    'consensus_medoid': consensus_medoid([a,b]),
}
decision_report = {name:{'rule':value.rule,'subproblems':len(value.decomposition.subproblems)} for name,value in decision_results.items()}
print('Metrics:', metric_result)
print('Decision operators:', json.dumps(decision_report, indent=2))
assert set(decision_results) == {'authority','rank_sum','majority','union','synthesis','compromise','consensus_medoid'}

In [ ]:
from sociodec.catalogue import STRATEGIES, CANDIDATE_STRATEGIES
from sociodec.e1_protocol import ALL_DATASETS
catalogue_report = {
  'paper_strategy_count': len(STRATEGIES),
  'paper_strategy_ids': [x.identifier.value for x in STRATEGIES],
  'unvalidated_candidate_count': len(CANDIDATE_STRATEGIES),
  'datasets_in_E1_protocol': list(ALL_DATASETS),
  'experiment_ids_defined_in_runner_contract': [f'E{i}' for i in range(1,10)],
}
print(json.dumps(catalogue_report, indent=2))
assert len(STRATEGIES) == 13 and len(CANDIDATE_STRATEGIES) == 5

In [ ]:
final_report = {
  'purpose': 't4_small_model_core_engineering_smoke_only',
  'scientific_result': False,
  'e1_e9_executed': False,
  'repository_sha': actual_sha,
  'gpu': gpu_name,
  'gpu_memory_gib': round(gpu_gib,3),
  'model_id': MODEL_CONFIG.model_id,
  'core': core_report,
  'metric_smoke': {'validity': metric_result.validity, 'coverage': metric_result.coverage, 'redundancy': metric_result.redundancy, 'coupling': metric_result.coupling},
  'decision_operators': decision_report,
  'catalogue': catalogue_report,
  'limitations': ['single small model only','no MA/MB/MC/ME comparison','no frozen paper benchmark data','no E1-E9 scientific trials','V3/V4 require an executor and are not evaluated by the structural validator'],
}
report_path = ARTIFACTS / 't4_small_model_core_smoke.json'
report_path.write_text(json.dumps(final_report, ensure_ascii=False, indent=2)+'\\n', encoding='utf-8')
print(json.dumps(final_report, ensure_ascii=False, indent=2))
print('Saved:', report_path)